In [4]:
vocab_size = 50257 # Vocabulary size
context_length = 1024 # Context length
emb_dim = 768 # Embedding dimension
n_heads = 12 # Number of attention heads
n_layers = 12 # Number of layers
drop_rate = 0.1 # Dropout rate

In [5]:
class DummyTransformerBlock(nn.Module):
    def __init__(self, vocab_size, context_length, emb_dim, n_heads, n_layers, drop_rate):
        super().__init__()
    def forward(self, x):
        return x
class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
    def forward(self, x):
        return x

# Base Model

In [1]:
import torch
import torch.nn as nn

class DummyGPTModel(nn.Module):
    def __init__(self, vocab_size, context_length, emb_dim, n_heads, n_layers, drop_rate):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.pos_embedding = nn.Embedding(context_length, emb_dim)
        self.transformer = DummyTransformerBlock(vocab_size, context_length, emb_dim, n_heads, n_layers, drop_rate)
        self.dropout = nn.Dropout(drop_rate)
        self.ff_layer = nn.Linear(emb_dim, vocab_size, bias=False)
        self.layer_normalization = DummyLayerNorm(emb_dim)

        
    def forward(self, inputs):
        emb = self.embedding(inputs)
        pos_embedding = self.pos_embedding(torch.arange(inputs.shape[-1]))
        x = emb + pos_embedding

        x = self.dropout(x)

        x = self.transformer(x)
        x = self.layer_normalization(x)

        return self.ff_layer(x)


In [2]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [6]:
torch.manual_seed(123)
model = DummyGPTModel(vocab_size, context_length, emb_dim, n_heads, n_layers, drop_rate)
logits = model(batch)
print("Output shape:", logits.shape)
print(logits)

Output shape: torch.Size([2, 4, 50257])
tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6754, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)


# Layer Normalization

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

# GELU

In [13]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
        torch.sqrt(torch.tensor(2.0 / torch.pi)) *
        (x + 0.044715 * torch.pow(x, 3))
        ))

# Feed Forward Network

In [11]:
class FeedForward(nn.Module):
    def __init__(self, emb_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(emb_size, emb_size * 4),
            GELU(),
            nn.Linear(emb_size * 4, emb_size)
        )
        

    def forward(self, x):
        output = self.network(x)
        return output
    

In [14]:
ffn = FeedForward(emb_dim)
x = torch.rand(2, 3, 768)
out = ffn(x)
print(out.shape)

torch.Size([2, 3, 768])


# Trasformer Block

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, emb_dim, in_dim, out_dim, n_heads, n_layers, drop_rate):
        super().__init__()
        self.attention = nn.Sequential(Attention() for _ in range(n_layers))
        self.dropout = nn.Dropout(drop_rate)
        self.feedfordward = FeedForward(emb_dim)
        self.layer_norm = LayerNorm(emb_dim)


    def forward(self, x):
        residual = x
        x = self.layer_norm(x)
        x = self.attention(x)
        x = self.dropout(x)

        x += residual
        residual = x

        x = self.layer_norm(x)
        x = self.feedfordward
        x = self.dropout(x)

        x += residual
        return x

# Whole Model

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, emb_dim, in_dim, out_dim, n_heads, n_layers, vocab_size, drop_rate):
        super().__init__()
        self.embeding = nn.Embedding(vocab_size, emb_dim)
        self.pos_embedding = nn.Embedding(context_length, emb_dim)
        self.dropout = nn.Dropout(drop_rate)
        self.backbone = nn.Sequential() 
        self.lm_head =  nn.Linear(emb_dim, vocab_size)
        self.layer_norm = LayerNorm(emb_dim)

    def forward(self, x):
        emb = self.embeding(x)
        pos_embedding = self.pos_embedding(torch.arange(x.shape[-1]))

        x = emb + pos_embedding
        x = self.dropout(x)

        x = self.backbone(x)
        x = self.layer_norm(x)
        logits = self.lm_head(x)

        return logits

        